# 🚀 Muse Spark Open Kernel Challenge ($100k)
### Official Submission Runner for NVIDIA H200 (Hopper SM90a)
**Participant:** Arunjyoti Sarma (IIT BHU Research Lab)
**Target Problem:** `muse_bf16_proj` (Dense Projections)

Simply click the Play button **`[ ▶ ]`** on each cell from top to bottom!

## Step 1: Verify Cloud GPU

In [ ]:
!nvidia-smi

## Step 2: Install `okc` CLI & Dependencies

In [ ]:
!curl -fsSL https://openkernelchallenge.dev/install.sh | sh
import os
os.environ["PATH"] = "/root/.okc/bin:" + os.environ["PATH"]
!pip install -q triton torch

## Step 3: Login to OKC with GitHub
1. If already logged in, this will verify your session instantly.
2. If not logged in, copy the 8-character code shown below, click **[github.com/login/device](https://github.com/login/device)**, and click Authorize.

In [ ]:
# Check if already authenticated
res = !okc problems 2>/dev/null
if any('muse_bf16_proj' in line for line in res):
    print('✓ You are ALREADY authenticated! You can proceed directly to Step 4.')
else:
    print('Initiating GitHub Device Login (auto-accepting terms)...')
    !printf 'y\n' | okc login


## Step 4: Initialize Problem Workspace (`muse_bf16_proj`)

In [ ]:
!okc init muse_bf16_proj

## Step 5: Deploy Our Optimized Hopper H200 Kernel to `muse_bf16_proj/submission.py`
This pre-packs weights and eliminates runtime slicing to beat SGLang's cuBLAS baseline.

In [ ]:
code = '''# okc: problem=muse_bf16_proj
"""Optimized Dense Projections for Muse Spark Open on Hopper H200.
Author: Arunjyoti Sarma (IIT BHU Research Lab)

Key Techniques:
1. Zero-Allocation Hot Path: Pre-slice and pre-contiguous all weight buffers during untimed prepare().
2. Pre-Transposed Layouts: Eliminate runtime matrix transpositions.
3. Dual-Path Router: Pre-transposed FP32 buffer for tokens <= 8, split BF16 GEMM for tokens > 8.
4. Native cuBLASLt Dispatch with Zero Slicing Overhead.
"""

from typing import TypedDict
import torch
import torch.nn.functional as F

try:
    from sglang.srt.layers.quantization.unquant import _bf16_gemm_dispatch_impl
    HAS_SGLANG = True
except Exception:
    HAS_SGLANG = False
    _bf16_gemm_dispatch_impl = None


class Weights(TypedDict):
    qkv_gate: torch.Tensor  # bfloat16 [2304, 8192]
    o: torch.Tensor         # bfloat16 [8192, 1024]
    pooler: torch.Tensor    # bfloat16 [4096, 8192]
    unpooler: torch.Tensor  # bfloat16 [8192, 4096]
    router: torch.Tensor    # float32 [256, 8192]


class Config(TypedDict):
    m: int
    n: int
    k: int
    layer: str


class Inputs(TypedDict):
    x: torch.Tensor
    config: Config


ROUTER_FP32_MAX_TOKENS = 8


def prepare(weights: Weights):
    state = {}
    for name, w in weights.items():
        w_contig = w.contiguous()
        state[name] = w_contig
        state[f"{name}_t"] = w_contig.t().contiguous()

    unpooler = weights["unpooler"]
    state["unpooler_1024"] = unpooler[:1024].contiguous()
    state["unpooler_1024_t"] = unpooler[:1024].t().contiguous()
    state["unpooler_8192"] = unpooler.contiguous()
    state["unpooler_8192_t"] = unpooler.t().contiguous()

    router = weights["router"]
    state["router_t"] = router.t().contiguous()
    hi = router.to(torch.bfloat16)
    lo = (router - hi.float()).to(torch.bfloat16)
    state["router_hi_t"] = hi.t().contiguous()
    state["router_lo_t"] = lo.t().contiguous()

    return state


def kernel_fn(data: Inputs, state: dict) -> torch.Tensor:
    x = data["x"]
    cfg = data["config"]
    layer = cfg["layer"]

    if layer == "router":
        if x.shape[0] <= ROUTER_FP32_MAX_TOKENS:
            return torch.mm(x.float(), state["router_t"])
        acc = torch.mm(x, state["router_hi_t"], out_dtype=torch.float32)
        acc += torch.mm(x, state["router_lo_t"], out_dtype=torch.float32)
        return acc

    if layer == "unpooler":
        n = cfg["n"]
        if n == 1024:
            w = state["unpooler_1024"]
            w_t = state["unpooler_1024_t"]
        elif n == 8192:
            w = state["unpooler_8192"]
            w_t = state["unpooler_8192_t"]
        else:
            w = state["unpooler"][:n]
            w_t = w.t().contiguous()

        if HAS_SGLANG and _bf16_gemm_dispatch_impl is not None:
            return _bf16_gemm_dispatch_impl(x, w, None)
        return torch.mm(x, w_t)

    w = state[layer]
    if HAS_SGLANG and _bf16_gemm_dispatch_impl is not None:
        return _bf16_gemm_dispatch_impl(x, w, None)
    return torch.mm(x, state[f"{layer}_t"])
'''
import os
os.makedirs('muse_bf16_proj', exist_ok=True)
with open('muse_bf16_proj/submission.py', 'w') as f:
    f.write(code)
print('✓ Successfully deployed optimized kernel to muse_bf16_proj/submission.py!')


## Step 6: Submit to Meta's Official H200 Cluster & Leaderboard! 🚀

In [ ]:
!okc submit muse_bf16_proj/submission.py